***

* [总目录](../0_Introduction/0_introduction.ipynb)
* [术语表](../0_Introduction/1_glossary.ipynb)
* [第 5 章：成像](5_0_introduction.ipynb)
    * 上一节：[5.0 引言](5_0_introduction.ipynb)
    * 下一节：[5.2 采样函数与点扩散函数](5_2_sampling_functions_and_psfs.ipynb)

***


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path
from IPython.display import HTML
%matplotlib inline
HTML('../style/course.css') #apply general CSS


导入本节所需的专用模块。


In [ ]:
HTML('../style/code_toggle.html')


## 5.1 空间频率<a id='imaging:sec:spatial'></a>

第 4 章给出的测量方程说明：在小视场近似下，可见度函数 $V(u,v)$ 与天空亮度分布 $I(l,m)$ 互为 Fourier 变换。因此，进入成像问题之前，必须先把一个概念建立稳固：`uv` 平面中的每个采样点并不是天空中的一个位置，而是天空亮度在某个空间频率上的一个复系数。

“空间频率”这个说法看上去抽象，但它描述的其实是图像在空间中变化得有多快。亮度缓慢起伏的大尺度结构对应低空间频率，边缘、细丝和小尺度团块对应高空间频率。干涉阵用长短不同、方向不同的基线去测量这些频率分量，最后再把它们组合回图像。于是，理解空间频率就是理解干涉成像为什么能在没有实体大口径天线的情况下恢复结构。


### 5.1.1 图像与可见度是一对 Fourier 对

在第 4 章的二维 Fourier 近似下，表观天空亮度与可见度满足

$$
V(u,v)=\int\!\!\int I(l,m)\,e^{-2\pi i(ul+vm)}\,dl\,dm,
$$

$$
I(l,m)=\int\!\!\int V(u,v)\,e^{+2\pi i(ul+vm)}\,du\,dv.
$$

这里 $(l,m)$ 是图像域中的角位置坐标，$(u,v)$ 是以波长为单位的空间频率坐标。指数核中的相位 $2\pi(ul+vm)$ 说明：每个可见度样本都对应一组在图像平面上起伏的复正弦条纹，而反变换就是把许多这样的条纹按复系数叠加起来，重新组成图像。

若图像已经在规则网格上采样，连续积分就退化为离散求和。对一个 $N\times M$ 像素的实值图像 $I[p,q]$，离散 Fourier 变换可写为

$$
V[k,r]=\sum_{p=0}^{N-1}\sum_{q=0}^{M-1} I[p,q]\,
\exp\!\left[-2\pi i\left(\frac{kp}{N}+\frac{rq}{M}\right)\right],
$$

$$
I[p,q]=\frac{1}{NM}\sum_{k=0}^{N-1}\sum_{r=0}^{M-1} V[k,r]\,
\exp\!\left[+2\pi i\left(\frac{kp}{N}+\frac{rq}{M}\right)\right].
$$

快速 Fourier 变换 FFT 做的正是这两组离散求和。`fftshift` 把数组中的零空间频率移到中心，便于显示；当代码用中心化的图像坐标解释相位时，还必须与 `ifftshift` 成对使用，不能把它当作不影响坐标约定的装饰操作。

还有两个性质在成像中经常反复出现。第一，若图像是实值的，则 Fourier 变换满足厄米对称性

$$
V(-u,-v)=V^*(u,v),
$$

这意味着频域中并不是每个像素都携带独立信息。第二，平移定理说明：图像中一个源离相位中心越远，对应的可见度相位变化越快。后面点源和双点源的例子，会把这两条性质具体地表现出来。


In [ ]:
fig_dir = Path('figures')
fig_dir.mkdir(exist_ok=True)


def rgb_to_gray(img):
    arr = img[..., :3].astype(float)
    if arr.max() > 1.5:
        arr /= 255.0
    return 0.2989 * arr[..., 0] + 0.5870 * arr[..., 1] + 0.1140 * arr[..., 2]


dish = rgb_to_gray(mpimg.imread('figures/synthetic_radio_dish_scene.png'))
galaxy = rgb_to_gray(mpimg.imread('figures/synthetic_spiral_galaxy.png'))

fft_dish = np.fft.fftshift(np.fft.fft2(dish))
fft_galaxy = np.fft.fftshift(np.fft.fft2(galaxy))

fig, axes = plt.subplots(2, 3, figsize=(13.5, 8.2))
items = [
    ('Dish scene', dish, 'gray', None),
    ('Dish amplitude', np.log10(np.abs(fft_dish) + 1e-6), 'magma', None),
    ('Dish phase', np.angle(fft_dish), 'twilight', (-np.pi, np.pi)),
    ('Synthetic galaxy', galaxy, 'gray', None),
    ('Galaxy amplitude', np.log10(np.abs(fft_galaxy) + 1e-6), 'magma', None),
    ('Galaxy phase', np.angle(fft_galaxy), 'twilight', (-np.pi, np.pi)),
]
for ax, (title, data, cmap, limits) in zip(axes.flat, items):
    im = ax.imshow(data, cmap=cmap)
    if limits is not None:
        im.set_clim(*limits)
    ax.set_title(title)
    ax.set_xticks([])
    ax.set_yticks([])
fig.tight_layout()
fig.savefig(fig_dir / 'spatial_frequency_image_spectra.png', dpi=180, bbox_inches='tight')
plt.close(fig)


![图像与频谱](figures/spatial_frequency_image_spectra.png)

**图 5.1.1** 两幅普通图像及其二维 Fourier 频谱。频谱中心聚集的强亮成分对应大尺度、缓慢变化的结构；远离中心的位置对应更精细的纹理和边缘。相位图在视觉上比幅度图更复杂，但其中保存了大量关于结构位置和轮廓的关键信息。


### 5.1.2 幅度控制尺度分布，相位控制结构位置

图 5.1.1 中最醒目的现象，是两幅图像的 Fourier 幅度都在中心附近最强。这并不神秘。自然图像和天文图像通常都含有较强的大尺度背景或平滑结构，因此低空间频率往往承载更高能量。真正决定“轮廓在哪里、边缘朝哪个方向展开”的，往往是相位而不是幅度。

这个直觉可以通过交换两幅图像的幅度和相位来演示。设一幅图像的 Fourier 变换写成

$$
V(u,v)=|V(u,v)|\,e^{i\phi(u,v)},
$$

那么把一幅图像的幅度与另一幅图像的相位重新组合，再做反 Fourier 变换，就能观察二者各自负责哪一部分信息。这个实验对射电干涉成像尤其重要，因为它提醒我们：即便幅度标定已经相当准确，相位误差仍然足以严重破坏结构恢复。不过，“结构跟随相位”是自然图像上的经验现象，不是幅度与相位可以独立分工的定理；二者共同决定图像，零幅度处的相位也没有定义。


In [ ]:
fig_dir = Path('figures')
fig_dir.mkdir(exist_ok=True)


def rgb_to_gray(img):
    arr = img[..., :3].astype(float)
    if arr.max() > 1.5:
        arr /= 255.0
    return 0.2989 * arr[..., 0] + 0.5870 * arr[..., 1] + 0.1140 * arr[..., 2]


dish = rgb_to_gray(mpimg.imread('figures/synthetic_radio_dish_scene.png'))
galaxy = rgb_to_gray(mpimg.imread('figures/synthetic_spiral_galaxy.png'))

fft_dish = np.fft.fftshift(np.fft.fft2(dish))
fft_galaxy = np.fft.fftshift(np.fft.fft2(galaxy))

amp_dish, phase_dish = np.abs(fft_dish), np.angle(fft_dish)
amp_galaxy, phase_galaxy = np.abs(fft_galaxy), np.angle(fft_galaxy)

hybrid_galaxy = np.fft.ifft2(np.fft.ifftshift(amp_dish * np.exp(1j * phase_galaxy))).real
hybrid_dish = np.fft.ifft2(np.fft.ifftshift(amp_galaxy * np.exp(1j * phase_dish))).real
phase_only_galaxy = np.fft.ifft2(np.fft.ifftshift(np.exp(1j * phase_galaxy))).real
amp_only_galaxy = np.fft.fftshift(np.fft.ifft2(np.fft.ifftshift(amp_galaxy))).real

fig, axes = plt.subplots(2, 2, figsize=(10.5, 10.0))
items = [
    ('phase: galaxy, amplitude: dish', hybrid_galaxy),
    ('phase: dish, amplitude: galaxy', hybrid_dish),
    ('galaxy reconstructed from phase only', phase_only_galaxy),
    ('galaxy reconstructed from amplitude only', amp_only_galaxy),
]
for ax, (title, data) in zip(axes.flat, items):
    ax.imshow(data, cmap='gray')
    ax.set_title(title)
    ax.set_xticks([])
    ax.set_yticks([])
fig.tight_layout()
fig.savefig(fig_dir / 'phase_amplitude_role.png', dpi=180, bbox_inches='tight')
plt.close(fig)


![幅度与相位的作用](figures/phase_amplitude_role.png)

**图 5.1.2** 上排交换合成星系与天线场景的幅度和相位，下排分别只保留星系图像的相位或只保留其幅度。代码保留反变换的正负实值，而不取绝对值掩盖符号；结构轮廓更强地跟随相位，只保留零相位幅度时则形成以中心为原点的对称响应。

这个现象背后的物理含义很直接。幅度决定不同空间频率成分各自有多强，相位决定这些成分叠加后在图像域中落在什么位置、以怎样的相干方式相长或相消。干涉测量中的相位误差之所以危险，正是因为结构定位对相位极其敏感。


### 5.1.3 单点源与平移定理

把讨论从复杂图像缩回到最简单的源模型，数学关系会变得十分透明。若图像中只有一个位于 $(l_0,m_0)$ 的点源，亮度为 $I_0$，则

$$
I(l,m)=I_0\,\delta(l-l_0)\,\delta(m-m_0).
$$

将它代入 Fourier 关系可得

$$
V(u,v)=I_0\,e^{-2\pi i(ul_0+vm_0)}.
$$

这个结果非常重要。第一，点源的可见度幅度处处相同，等于源强 $I_0$，因此单个点源在频域中是“铺满整个平面”的。第二，点源偏离相位中心只会改变相位，不会改变幅度。第三，等相位线满足 $ul_0+vm_0=\text{常数}$，因此频域中的条纹方向总是与源偏移方向正交。源离中心越远，条纹越密，相位变化越快。


In [ ]:
fig_dir = Path('figures')
fig_dir.mkdir(exist_ok=True)


def point_source_fft(size, ypos, xpos, amp=1.0):
    img = np.zeros((size, size), dtype=float)
    img[ypos, xpos] = amp
    fft_img = np.fft.fftshift(np.fft.fft2(np.fft.ifftshift(img)))
    return img, fft_img


size = 129
center = size // 2
cases = [
    ('centered source', center, center),
    ('offset in $l$', center, center + 16),
    ('offset in $m$', center - 16, center),
]

fig, axes = plt.subplots(2, 3, figsize=(12.5, 7.2))
for col, (title, y0, x0) in enumerate(cases):
    img, fft_img = point_source_fft(size, y0, x0)
    axes[0, col].imshow(img, cmap='gray', interpolation='nearest')
    axes[0, col].set_title(title)
    axes[0, col].set_xticks([])
    axes[0, col].set_yticks([])

    phase = np.angle(fft_img)
    axes[1, col].imshow(phase, cmap='twilight', vmin=-np.pi, vmax=np.pi)
    axes[1, col].set_title('visibility phase')
    axes[1, col].set_xticks([])
    axes[1, col].set_yticks([])

fig.tight_layout()
fig.savefig(fig_dir / 'point_source_phase_ramps.png', dpi=180, bbox_inches='tight')
plt.close(fig)


![点源与相位斜坡](figures/point_source_phase_ramps.png)

**图 5.1.3** 单点源对应的可见度相位图。位于相位中心的点源给出平坦相位；沿 $l$ 或 $m$ 方向偏移后，相位变成沿相应正交方向起伏的条纹。条纹越密，说明源离相位中心越远。


### 5.1.4 双点源、条纹振荡与可见度零点

两个点源时，可见度不再是单个复指数，而是多个复指数的线性叠加。若两个点源位于 $(l_1,0)$ 和 $(l_2,0)$，强度分别为 $I_1$ 与 $I_2$，则沿 $v=0$ 的截面有

$$
V(u,0)=I_1 e^{-2\pi i u l_1}+I_2 e^{-2\pi i u l_2}.
$$

若两个源等强并关于相位中心对称，即 $I_1=I_2=I_0$、$l_1=-\Delta l/2$、$l_2=+\Delta l/2$，则式子化简为

$$
V(u,0)=2I_0\cos(\pi u\Delta l).
$$

这就是干涉测量中最基本的条纹振荡之一。源间距 $\Delta l$ 越大，幅度振荡越快；当

$$
u_n = \frac{n+1/2}{\Delta l}, \qquad n=0,1,2,\ldots
$$

时，可见度幅度出现零点。零点的位置因此直接编码了源的角分离尺度。若两源不等强，完全相消不再可能，幅度零点会被抬起。若两源等强但整体平移到中心 $l_c=(l_1+l_2)/2$，则

$$
V(u,0)=2I_0e^{-2\pi iul_c}\cos(\pi u\Delta l).
$$

平移只增加线性相位，不会改变幅度和零点位置。只有强度比、源结构或分离尺度改变时，幅度曲线才会相应改变。


In [ ]:
fig_dir = Path('figures')
fig_dir.mkdir(exist_ok=True)

u = np.linspace(-80.0, 80.0, 1601)

def visibility_pair(u_coord, l1, l2, i1, i2):
    return i1 * np.exp(-2j * np.pi * u_coord * l1) + i2 * np.exp(-2j * np.pi * u_coord * l2)

v_sym = visibility_pair(u, -0.035, 0.035, 1.0, 1.0)
v_shifted = visibility_pair(u, -0.020, 0.050, 1.0, 1.0)
v_unequal = visibility_pair(u, -0.020, 0.050, 1.0, 0.55)

size = 129
center = size // 2
img_sym = np.zeros((size, size), dtype=float)
img_sym[center, center - 18] = 1.0
img_sym[center, center + 18] = 1.0
img_shifted = np.zeros((size, size), dtype=float)
img_shifted[center, center - 10] = 1.0
img_shifted[center, center + 26] = 1.0
img_unequal = img_shifted.copy()
img_unequal[center, center + 26] = 0.55

fig, axes = plt.subplots(3, 3, figsize=(13.0, 10.5))

axes[0, 0].imshow(img_sym, cmap='gray', interpolation='nearest')
axes[0, 0].set_title('equal symmetric pair')
axes[1, 0].imshow(img_shifted, cmap='gray', interpolation='nearest')
axes[1, 0].set_title('equal shifted pair')
axes[2, 0].imshow(img_unequal, cmap='gray', interpolation='nearest')
axes[2, 0].set_title('unequal shifted pair')
for ax in axes[:, 0]:
    ax.set_xticks([])
    ax.set_yticks([])

axes[0, 1].plot(u, np.abs(v_sym), lw=2, color='#1d3557')
axes[0, 1].set_title(r'$|V(u,0)|$')
axes[0, 1].set_ylabel('equal pair')
axes[0, 1].grid(alpha=0.3)

axes[1, 1].plot(u, np.abs(v_shifted), lw=2, color='#1d3557')
axes[1, 1].set_ylabel('equal shifted pair')
axes[1, 1].grid(alpha=0.3)
axes[2, 1].plot(u, np.abs(v_unequal), lw=2, color='#1d3557')
axes[2, 1].set_xlabel(r'$u$ [wavelengths]')
axes[2, 1].set_ylabel('unequal pair')
axes[2, 1].grid(alpha=0.3)

for row, vis in enumerate([v_sym, v_shifted, v_unequal]):
    phase = np.unwrap(np.angle(vis))
    axes[row, 2].plot(u, phase, lw=2, color='#d62828')
    axes[row, 2].set_title('phase')
    axes[row, 2].grid(alpha=0.3)
axes[2, 2].set_xlabel(r'$u$ [wavelengths]')

fig.tight_layout()
fig.savefig(fig_dir / 'double_source_visibility.png', dpi=180, bbox_inches='tight')
plt.close(fig)


![双点源的可见度](figures/double_source_visibility.png)

**图 5.1.4** 双点源的可见度不再是单一相位斜坡，而是多个复指数的叠加。上、中两行具有相同强度和间距，因此幅度曲线及零点完全相同；中行的整体平移只叠加相位斜坡。下行改变强度比后不能完全相消，零点才被抬起。

这个例子与综合孔径分辨率直接相连：采样到明显的幅度振荡乃至第一个零点，会为双源分离提供很强约束。但“必须测到第一个零点才能分辨”并不是普适判据；在高信噪比且源模型可信时，可以从较短基线上的细微变化估计小于名义波束的分离，而不完备采样和模型失配会限制这种超分辨推断。


### 5.1.5 从频域采样回到图像域

前面所有例子都在说明一件事：图像可以看成许多空间频率成分的叠加。于是，若在频域中只保留其中一部分成分，再做反 Fourier 变换，就能直接观察“不完备采样”对图像的影响。这个演示仍然是理想化的，因为它使用的是规则 Fourier 网格，而真实干涉仪给出的则是不规则的 `uv` 采样点；但它足以说明为什么缺失频率会在图像域中留下系统性的伪影。


In [ ]:
fig_dir = Path('figures')
fig_dir.mkdir(exist_ok=True)


def rgb_to_gray(img):
    arr = img[..., :3].astype(float)
    if arr.max() > 1.5:
        arr /= 255.0
    return 0.2989 * arr[..., 0] + 0.5870 * arr[..., 1] + 0.1140 * arr[..., 2]


galaxy = rgb_to_gray(mpimg.imread('figures/synthetic_spiral_galaxy.png'))
galaxy = galaxy[::2, ::2]
vis = np.fft.fft2(galaxy)

rng = np.random.default_rng(7)
counts = [64, 512, 4096, 32768]
fig, axes = plt.subplots(2, 4, figsize=(13.5, 6.8))

for col, nsamp in enumerate(counts):
    mask = np.zeros_like(vis, dtype=bool)
    flat_index = rng.choice(vis.size, size=nsamp // 2, replace=False)
    y, x = np.unravel_index(flat_index, vis.shape)
    conjugate = np.ravel_multi_index(((-y) % vis.shape[0], (-x) % vis.shape[1]), vis.shape)
    mask.flat[np.concatenate([flat_index, conjugate, [0]])] = True
    sampled_vis = vis * mask
    recon = np.fft.ifft2(sampled_vis).real

    axes[0, col].imshow(np.fft.fftshift(mask), cmap='gray', origin='lower')
    axes[0, col].set_title(f'{np.count_nonzero(mask)} Fourier samples')
    axes[0, col].set_xticks([])
    axes[0, col].set_yticks([])

    axes[1, col].imshow(recon, cmap='gray', origin='lower')
    axes[1, col].set_xticks([])
    axes[1, col].set_yticks([])
    axes[1, col].set_title('reconstructed image')

fig.tight_layout()
fig.savefig(fig_dir / 'visibility_sampling_reconstruction.png', dpi=180, bbox_inches='tight')
plt.close(fig)


![频域采样与重建](figures/visibility_sampling_reconstruction.png)

**图 5.1.5** 在规则 Fourier 网格上随机保留不同数量的频率样本，并同时保留每个样本的厄米共轭点，再做反 Fourier 变换。共轭配对保证实值天空仍重建为实值图像；采样点很少时，图像更像若干低对比度条纹和模糊块的叠加，采样增加后主要结构才逐步显现。

这个演示已经隐含了脏图像的基本逻辑。缺失的不是某几个像素，而是整片频率区域；因此重建误差也不是局部噪声，而是由采样函数决定的全局卷积效应。第 5.2 节将把这种“由采样导致的结构性伪影”写成明确的数学形式，并由此引入点扩散函数和脏波束。


***

* 下一节：[5.2 采样函数与点扩散函数](5_2_sampling_functions_and_psfs.ipynb)
